# Live2D Prompt-Bank Inference

This notebook runs SAM3 image inference over a Live2D canonical prompt bank. It scans one character image class by class, deduplicates masks within each class, then writes an overlay PNG and COCO-style RLE masks.

In [ ]:
from pathlib import Path
import contextlib
import json
import math
import random

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from pycocotools import mask as mask_utils

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'sam3').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

TAXONOMY_PATH = REPO_ROOT / 'scripts' / 'live2d' / 'live2d_canonical_taxonomy_hair_merged.json'
CHECKPOINT_PATH = REPO_ROOT / 'sam3.pt'
BPE_PATH = REPO_ROOT / 'sam3' / 'assets' / 'bpe_simple_vocab_16e6.txt.gz'

# Change this to any single rendered character image.
IMAGE_PATH = REPO_ROOT / 'dataset' / 'live2d_parts_canonical_hair_merged' / 'val' / '地雷系 十六夜咲夜_3e81bb24.png'
OUTPUT_DIR = REPO_ROOT / 'runs' / 'live2d_prompt_inference_hair_merged'
PER_CLASS_THRESHOLDS_PATH = REPO_ROOT / 'runs' / 'live2d_threshold_sweep' / 'trainer_v2_hair_merged_e25_iou050_nms050_per_class_thresholds.json'

RUN_SAM3_INFERENCE = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def inference_precision_context():
    if DEVICE == 'cuda':
        return torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    return contextlib.nullcontext()

CONFIDENCE_THRESHOLD = 0.8
USE_PER_CLASS_THRESHOLDS = True
PER_CLASS_THRESHOLD_FLOOR = 0.8
MAX_ALIASES_PER_CLASS = 4
IOU_DEDUP_THRESHOLD = 0.5
MIN_MASK_AREA = 16

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('repo:', REPO_ROOT)
print('device:', DEVICE)
print('image:', IMAGE_PATH)
print('taxonomy:', TAXONOMY_PATH)


In [ ]:
taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding='utf-8'))
categories = taxonomy['categories']

def build_prompt_bank(categories, max_aliases=4):
    bank = {}
    for category in categories:
        prompts = []
        for value in [category['name'], *category.get('aliases', [])]:
            value = str(value).replace('_', ' ').strip()
            if value and value not in prompts:
                prompts.append(value)
        ascii_prompts = [p for p in prompts if p.isascii()]
        non_ascii_prompts = [p for p in prompts if not p.isascii()]
        ordered = ascii_prompts + non_ascii_prompts
        bank[category['name']] = ordered[:max_aliases] if max_aliases else ordered
    return bank

prompt_bank = build_prompt_bank(categories, MAX_ALIASES_PER_CLASS)
for name, prompts in prompt_bank.items():
    print(f'{name:14s}: {prompts}')


In [ ]:
image = Image.open(IMAGE_PATH).convert('RGB')
width, height = image.size
print(width, height)
display(image)


In [ ]:
def tensor_to_numpy(value):
    if value is None:
        return None
    if isinstance(value, torch.Tensor):
        value = value.detach()
        if value.dtype == torch.bfloat16:
            value = value.float()
        return value.cpu().numpy()
    return np.asarray(value)

def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union else 0.0

def dedupe_predictions(predictions, threshold=0.85):
    kept = []
    for pred in sorted(predictions, key=lambda x: x['score'], reverse=True):
        same_class = [p for p in kept if p['category_id'] == pred['category_id']]
        if any(mask_iou(pred['mask'], p['mask']) >= threshold for p in same_class):
            continue
        kept.append(pred)
    return kept

def mask_to_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return [0.0, 0.0, 0.0, 0.0]
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [float(x0), float(y0), float(x1 - x0 + 1), float(y1 - y0 + 1)]

def encode_rle(mask):
    encoded = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    encoded['counts'] = encoded['counts'].decode('ascii')
    return encoded


def load_per_class_thresholds(path):
    if not USE_PER_CLASS_THRESHOLDS or not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding='utf-8'))
    return {
        str(name): float(value)
        for name, value in payload.get('category_thresholds_by_name', {}).items()
    }

PER_CLASS_THRESHOLDS_BY_NAME = load_per_class_thresholds(PER_CLASS_THRESHOLDS_PATH)
print('per-class threshold file:', PER_CLASS_THRESHOLDS_PATH)
print('loaded per-class thresholds:', PER_CLASS_THRESHOLDS_BY_NAME)

def score_threshold_for(category_name):
    if not USE_PER_CLASS_THRESHOLDS:
        return CONFIDENCE_THRESHOLD
    learned = PER_CLASS_THRESHOLDS_BY_NAME.get(category_name, CONFIDENCE_THRESHOLD)
    return max(PER_CLASS_THRESHOLD_FLOOR, float(learned))

def filter_predictions_by_score(predictions):
    return [
        pred for pred in predictions
        if pred['score'] >= score_threshold_for(pred['category_name'])
    ]

def state_to_predictions(state, category, prompt):
    masks = tensor_to_numpy(state.get('masks'))
    boxes = tensor_to_numpy(state.get('boxes'))
    scores = tensor_to_numpy(state.get('scores'))
    if masks is None or len(masks) == 0:
        return []
    masks = np.asarray(masks).astype(bool)
    if masks.ndim == 4:
        masks = masks[:, 0]
    if boxes is None:
        boxes = np.array([mask_to_bbox(mask) for mask in masks], dtype=np.float32)
    if scores is None:
        scores = np.ones((len(masks),), dtype=np.float32)
    preds = []
    for mask, box, score in zip(masks, boxes, scores):
        area = int(mask.sum())
        if area < MIN_MASK_AREA:
            continue
        preds.append({
            'category_id': int(category['id']),
            'category_name': category['name'],
            'prompt': prompt,
            'score': float(score),
            'bbox_xyxy': [float(x) for x in box.tolist()],
            'mask': mask,
            'area': area,
        })
    return preds


In [ ]:
if RUN_SAM3_INFERENCE:
    from sam3 import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor

    model = build_sam3_image_model(
        bpe_path=str(BPE_PATH),
        checkpoint_path=str(CHECKPOINT_PATH),
        load_from_HF=False,
        device=DEVICE,
        eval_mode=True,
        enable_segmentation=True,
    )
    processor = Sam3Processor(model, device=DEVICE, confidence_threshold=CONFIDENCE_THRESHOLD)
    with inference_precision_context():
        inference_state = processor.set_image(image)
    print('SAM3 loaded')
else:
    processor = None
    inference_state = None
    print('Dry run only. Set RUN_SAM3_INFERENCE = True to load SAM3 and scan prompts.')


In [ ]:
all_predictions = []

if RUN_SAM3_INFERENCE:
    with torch.inference_mode(), inference_precision_context():
        for category in categories:
            class_predictions = []
            for prompt in prompt_bank[category['name']]:
                processor.reset_all_prompts(inference_state)
                state = processor.set_text_prompt(prompt=prompt, state=inference_state)
                preds = state_to_predictions(state, category, prompt)
                class_predictions.extend(preds)
                print(f'{category["name"]:14s} | {prompt:24s} | {len(preds)} masks')
            class_predictions = dedupe_predictions(class_predictions, IOU_DEDUP_THRESHOLD)
            class_predictions = filter_predictions_by_score(class_predictions)
            all_predictions.extend(class_predictions)
            print(f'  kept {len(class_predictions)} masks for {category["name"]}')

all_predictions = dedupe_predictions(all_predictions, IOU_DEDUP_THRESHOLD)
all_predictions = filter_predictions_by_score(all_predictions)
print('total predictions:', len(all_predictions))


In [ ]:
def predictions_to_coco(predictions, image_path, width, height, categories):
    annotations = []
    for ann_id, pred in enumerate(predictions, start=1):
        mask = pred['mask']
        rle = encode_rle(mask)
        bbox = mask_to_bbox(mask)
        annotations.append({
            'id': ann_id,
            'image_id': 1,
            'category_id': pred['category_id'],
            'segmentation': rle,
            'area': int(mask.sum()),
            'bbox': bbox,
            'iscrowd': 0,
            'score': pred['score'],
            'attributes': {
                'category_name': pred['category_name'],
                'prompt': pred['prompt'],
            },
        })
    return {
        'images': [{
            'id': 1,
            'file_name': Path(image_path).name,
            'width': width,
            'height': height,
        }],
        'annotations': annotations,
        'categories': [
            {'id': int(c['id']), 'name': c['name'], 'supercategory': 'live2d_parts'}
            for c in categories
        ],
    }

def color_for_category(category_id):
    rng = random.Random(category_id * 1009)
    return tuple(rng.randint(40, 235) for _ in range(3))

def render_overlay(image, predictions, alpha=0.46):
    base = image.convert('RGBA')
    overlay = Image.new('RGBA', base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    for pred in sorted(predictions, key=lambda x: x['area'], reverse=True):
        color = color_for_category(pred['category_id'])
        mask_img = Image.fromarray((pred['mask'] * int(255 * alpha)).astype(np.uint8), mode='L')
        fill = Image.new('RGBA', base.size, (*color, 0))
        fill.putalpha(mask_img)
        overlay = Image.alpha_composite(overlay, fill)
    out = Image.alpha_composite(base, overlay)
    draw = ImageDraw.Draw(out)
    for pred in predictions:
        x, y, w, h = mask_to_bbox(pred['mask'])
        color = color_for_category(pred['category_id'])
        draw.rectangle([x, y, x + w, y + h], outline=color, width=2)
        label = f"{pred['category_name']} {pred['score']:.2f}"
        draw.text((x + 2, y + 2), label, fill=(255, 255, 255, 255), stroke_width=2, stroke_fill=(0, 0, 0, 220))
    return out.convert('RGB')


In [ ]:
coco = predictions_to_coco(all_predictions, IMAGE_PATH, width, height, categories)
coco_path = OUTPUT_DIR / f'{IMAGE_PATH.stem}_sam3_live2d_predictions.coco.json'
overlay_path = OUTPUT_DIR / f'{IMAGE_PATH.stem}_sam3_live2d_overlay.png'
summary_path = OUTPUT_DIR / f'{IMAGE_PATH.stem}_sam3_live2d_prompt_summary.json'

coco_path.write_text(json.dumps(coco, ensure_ascii=False, indent=2), encoding='utf-8')
summary = [
    {k: v for k, v in pred.items() if k != 'mask'}
    for pred in all_predictions
]
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

overlay = render_overlay(image, all_predictions)
overlay.save(overlay_path)

print('wrote:', coco_path)
print('wrote:', overlay_path)
print('wrote:', summary_path)
display(overlay)
